# H1L1 — Attention, step by step

**Notebook 1 of this week.** We compute attention by hand for three tokens,
with vectors of dimension 2, so that every number can be checked on paper.
The parts follow the slides: after each block of slides we run the matching
part here.

1. the running example: the tokens and their query, key and value vectors
2. the scores
3. the softmax
4. the weighted sum of the values
5. a later layer: the last token reads what the first layer delivered
6. the MLP: a fact from a combination, and the next word
7. all tokens at once, as matrices

**Colab:** Runtime → Run all. The CPU runtime is enough; this takes seconds.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

C_BLUE, C_RED, C_TEAL, C_PURPLE, C_GRAY = \
    "#1F4E79", "#C00000", "#0E7C7B", "#7030A0", "#595959"
FIGDIR = os.environ.get("SEMINAR_FIGDIR")          # set only when the slides are built


def save(fig, name):
    """Only when the slides are built: store the figure for the deck."""
    if FIGDIR:
        for ext in ("pdf", "png"):
            fig.savefig(os.path.join(FIGDIR, f"{name}.{ext}"), bbox_inches="tight", dpi=200)


## Part 1 — The running example

The text is "France, its capital is", and the model must predict the next
word. The answer depends on *France*, three tokens back. Attention delivers it
in two steps. In the first layer, *capital* takes in information from
*France*. In a later layer, possibly several layers after the first, *is*
takes in information from *capital*.
Parts 1 to 4 compute the first layer for the token *capital*, which can read
*France*, *its* and itself.

Each token has three vectors. The **query** $q$ describes what the token is
looking for, the **key** $k$ describes what it contains, and the **value** $v$
is the information it passes on when it is selected. In a trained model these
vectors are computed from the token by three learned matrices. Here we write
them down directly and give the two dimensions a meaning, so that the numbers
can be read:

| dimension | meaning (chosen for this example) |
|---|---|
| 0 | "is a country" |
| 1 | "is a function word" |

In a trained model the dimensions do not have such simple meanings; the
arithmetic is the same.


In [ ]:
TOKENS = ["France", "its", "capital"]
K = np.array([[2., 0.],    # France
              [0., 2.],    # its
              [1., 1.]])   # capital
V = np.array([[1., 0.],
              [0., 1.],
              [1., 1.]])
q_capital = np.array([1., 0.])   # looks for a country

for t, k, v in zip(TOKENS, K, V):
    print(f"{t:8s} k={k} v={v}")

fig, ax = plt.subplots(figsize=(4.4, 3.8))
for t, k, c in zip(TOKENS, K, [C_BLUE, C_GRAY, C_TEAL]):
    ax.annotate("", xy=k, xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=c, lw=2))
    ax.text(k[0] + 0.06, k[1] + 0.06, f"key of {t}", color=c)
ax.annotate("", xy=q_capital, xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color=C_RED, lw=2, linestyle="--"))
ax.text(0.5, -0.28, "query of capital", color=C_RED, ha="center")
ax.set_xlim(-0.3, 2.9); ax.set_ylim(-0.45, 2.4); ax.set_aspect("equal")
ax.set_xlabel("dimension 0: a country"); ax.set_ylabel("dimension 1: a function word")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save(fig, "fig-t-w5-h1l1-p1"); plt.show()


## Part 2 — The scores

The query of *capital* is compared with the key of every token by a dot
product. A large dot product means that the key contains what the query looks
for. The query $[1, 0]$ looks only along dimension 0, "is a country", so the
score of each token is simply how much of a country it is.

The scores are then divided by $\sqrt{d_k}$, the square root of the
dimension of the keys, here $\sqrt 2$. This keeps the scores in a range in
which the softmax works well; we return to the reason later.


In [ ]:
d_k = K.shape[1]
dots = K @ q_capital
scores = dots / np.sqrt(d_k)

for t, d, s in zip(TOKENS, dots, scores):
    print(f"{t:8s} q·k={d:.1f}  score={s:.2f}")

fig, ax = plt.subplots(figsize=(4.4, 3.0))
ax.bar(TOKENS, scores, color=[C_BLUE, C_GRAY, C_TEAL])
for i, s in enumerate(scores):
    ax.text(i, s + 0.03, f"{s:.2f}", ha="center")
ax.set_ylabel("score = q·k / √2"); ax.set_ylim(0, 1.65)
ax.set_title("the scores of the query of 'capital'", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save(fig, "fig-t-w5-h1l1-p2"); plt.show()


**Exercise.** Before running Part 3, predict which token will receive the largest weight, and whether *its* will receive any weight at all.

<details><summary>Answer</summary>

*France* receives the largest weight, 0.58, because its key matches the query best. *its* receives 0.14: its score is 0, and $e^0 = 1$ is positive, so the softmax gives it a positive weight.

</details>


## Part 3 — The softmax

The softmax turns the scores into weights that are positive and sum to 1:

$$w_j = \frac{e^{s_j}}{\sum_{j'} e^{s_{j'}}}$$

*France* receives the largest weight, because its key matched the query best.
*its* still receives some weight: the softmax never gives exactly zero to a
finite score.


In [ ]:
def softmax(z):
    z = z - z.max(-1, keepdims=True)       # does not change the result; avoids overflow
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)

w = softmax(scores)

for t, s, wi in zip(TOKENS, scores, w):
    print(f"{t:8s} e^s={np.exp(s):.2f}  w={wi:.2f}")
print("sum of the weights:", w.sum())

fig, ax = plt.subplots(figsize=(4.4, 3.0))
ax.bar(TOKENS, w, color=[C_BLUE, C_GRAY, C_TEAL])
for i, wi in enumerate(w):
    ax.text(i, wi + 0.015, f"{wi:.2f}", ha="center")
ax.set_ylabel("weight"); ax.set_ylim(0, 0.7)
ax.set_title("the weights after the softmax (sum 1)", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save(fig, "fig-t-w5-h1l1-p3"); plt.show()


## Part 4 — The weighted sum of the values

The output for *capital* is the weighted sum of the values:

$$\mathrm{out} = 0.58\,v_{\text{France}} + 0.14\,v_{\text{its}} + 0.28\,v_{\text{capital}}$$

The result is $[0.86, 0.42]$. Its first coordinate, "is a country", is large:
after attention, the representation of *capital* carries the information
that the sentence is about a country, taken mostly from *France*. A later
layer can use this to predict "Paris".


In [ ]:
out = w @ V
print("out =", out.round(2))

fig, ax = plt.subplots(figsize=(6.4, 3.0))
bottom = np.zeros(2)
for t, wi, v, c in zip(TOKENS, w, V, [C_BLUE, C_GRAY, C_TEAL]):
    ax.bar([0, 1], wi * v, bottom=bottom, color=c, label=f"{wi:.2f} × value of {t}")
    bottom += wi * v
ax.set_xticks([0, 1], ["dim 0: country", "dim 1: function word"])
ax.set_ylabel("output")
ax.set_title("the output of 'capital' as a sum of weighted values")
ax.legend(frameon=False, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save(fig, "fig-t-w5-h1l1-p4"); plt.show()


**Exercise.** Compute $0.58\,[1,0] + 0.14\,[0,1] + 0.28\,[1,1]$ by hand.

<details><summary>Answer</summary>

$[0.58 + 0.28,\ 0.14 + 0.28] = [0.86,\ 0.42]$.

</details>


## Part 5 — A later layer

After the first layer, the output of *capital* is added to its vector, and
that vector is passed on through the following layers. In a later layer, which
need not be the next one, *is* looks for the word it describes, and the key of
*capital* matches. The values in that layer are the vectors as they arrive
there. We show two of their dimensions: "a country", which *capital* now
carries from *France* ($0.86$, the first coordinate of Part 4), and "a
capital", which is the meaning of *capital* itself.

For comparison we repeat the later layer without the first: *capital* then
carries no country, and neither does the output of *is*. We call the two
layers "layer 1" and "layer 2" below; any layers in between only pass the
vectors on.


In [ ]:
TOKENS2 = ["France", "its", "capital", "is"]
K2 = np.array([[1., 0.], [0., 0.],
               [0., 2.], [0., .5]])
# values: dims country, capital;
# row 3 is capital after layer 1
V2 = np.array([[1., 0.],
               [0., 0.],
               [out[0], 1.],
               [0., 0.]])
# the query of is: its subject
q_is = np.array([0., 1.])
w2 = softmax(K2 @ q_is / np.sqrt(2))
out_is = w2 @ V2

V2_no = V2.copy()
V2_no[2, 0] = 0.               # without layer 1
out_no = w2 @ V2_no

for t, wi in zip(TOKENS2, w2):
    print(f"{t:8s} w={wi:.2f}")
print("is, two layers: ", out_is.round(2))
print("is, layer 2 only:", out_no.round(2))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(8.4, 3.0))
a1.bar(TOKENS2, w2, color=[C_BLUE, C_GRAY, C_TEAL, C_PURPLE])
for i, wi in enumerate(w2):
    a1.text(i, wi + 0.015, f"{wi:.2f}", ha="center")
a1.set_ylim(0, 0.68); a1.set_title("layer 2: the weights of 'is'", fontsize=10)
x = np.arange(2)
a2.bar(x - 0.18, out_is, 0.36, color=C_RED, label="two layers")
a2.bar(x + 0.18, out_no, 0.36, color="#BBBBBB", label="layer 2 only")
a2.set_xticks(x, ["dim 0: country", "dim 1: capital"])
a2.set_ylim(0, 0.9)
a2.set_title("the output of 'is'", fontsize=10)
a2.legend(frameon=False, fontsize=8, ncol=2, loc="upper center")
for ax in (a1, a2):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save(fig, "fig-t-w5-h1l1-p5"); plt.show()


With both layers, the output of *is* carries both "a capital" and "a
country", the information needed to predict "Paris". With the later layer
alone it carries "a capital" but almost no country. The remaining layers and
the output layer then turn the vector of *is* into a probability for every
word of the vocabulary; in a trained model "Paris" receives the highest. Hour 3
finds the same two-step pattern, the induction head, in a trained model.


**Exercise.** Change the query of *is* to $[1, 0]$, so that it looks for a country. Which token does it read now, and does it still need the first layer?

<details><summary>Answer</summary>

The weights become 0.40 for *France* and 0.20 for each of the other three tokens, and the output of *is* is $[0.57,\ 0.20]$. *is* now reads *France* directly and gets the country without the first layer, but it loses "a capital" (0.20 instead of 0.55). To predict "Paris" it needs both, which it gets only by reading *capital* after *capital* has read *France*.

</details>


## Part 6 — The MLP, and the next word

After attention, the vector of *is* holds "a country" (France) and "a
capital". None of the tokens contains "Paris", and attention cannot create it:
its output is a weighted average of vectors that are already in the text. The
fact that the capital of France is Paris has to come from somewhere else. In a
transformer it comes from the **MLP**, which processes each position on its
own:

$$h = \mathrm{ReLU}(W_1 v + b_1)$$

Each hidden unit tests a combination of features, and the ReLU lets it fire
only when the combination is present. We write two units by hand:

| unit | computes | fires when |
|---|---|---|
| "Paris" | $\mathrm{ReLU}(2\,\text{country} + 2\,\text{capital} - 1.5)$ | a capital **and** France |
| "London" | $\mathrm{ReLU}(2\,\text{capital} - 2\,\text{country} - 0.5)$ | a capital, but **not** France |

The output layer then compares the vector with one output vector per word;
here the output vector of *Paris* reads the "Paris" unit, and that of *London*
the "London" unit. As in Part 5, we compare the vector of *is* with both
layers and with the later layer only.


In [ ]:
# row 0: unit "Paris", country and capital
# row 1: unit "London", capital, not country
W1 = np.array([[2., 2.], [-2., 2.]])
b1 = np.array([-1.5, -0.5])
relu = lambda z: np.maximum(0, z)

h_two = relu(W1 @ out_is + b1)   # both
h_one = relu(W1 @ out_no + b1)   # later only

WORDS = ["Paris", "London", "Lyon", "the"]
U = np.array([[3., 0.], [0., 3.],   # output vectors over the two units
              [0., 0.], [0., 0.]])
p_two, p_one = softmax(U @ h_two), softmax(U @ h_one)

print("MLP units, both layers:", h_two.round(2))
print("MLP units, later only: ", h_one.round(2))
for wd, a, b in zip(WORDS, p_two, p_one):
    print(f"{wd:7s} both {a:.2f}  later only {b:.2f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.0, 3.2))
x = np.arange(2)
a1.bar(x - 0.18, h_two, 0.36, color=C_RED, label="both layers")
a1.bar(x + 0.18, h_one, 0.36, color="#BBBBBB", label="later layer only")
a1.set_xticks(x, ['unit "Paris"', 'unit "London"']); a1.set_ylim(0, 1)
a1.set_title("the MLP units at 'is'", fontsize=10); a1.legend(frameon=False, fontsize=8)
x = np.arange(len(WORDS))
a2.bar(x - 0.18, p_two, 0.36, color=C_RED)
a2.bar(x + 0.18, p_one, 0.36, color="#BBBBBB")
for i, a in enumerate(p_two):
    a2.text(i - 0.18, a + 0.015, f"{a:.2f}", ha="center", fontsize=9)
a2.set_xticks(x, WORDS); a2.set_ylim(0, 1)
a2.set_title("the next word after 'France, its capital is'", fontsize=10)
for ax in (a1, a2):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save(fig, "fig-t-w5-h1l1-p6"); plt.show()


With both layers, the "Paris" unit fires and *Paris* receives the highest
probability. With the later layer only, the vector of *is* says "a capital"
but not "France", the "London" unit fires instead, and the model guesses
*London*. The two parts of a block divide the work: **attention moves
information between positions**; **the MLP computes new features at each
position**, here a fact looked up from a combination. Neither could produce
"Paris" alone.

Both are lookups, of two kinds. The **values** of attention come from the
tokens of the text being read: they are what this text says. The **MLP**
looks up patterns stored in its weights during training: what the model
knows. For "In this story, the capital of Freedonia is Blorp. ... The capital
of Freedonia is", the MLP knows nothing about Freedonia, and the answer must
be fetched by attention from the value of *Blorp*; for "The capital of France
is", no token contains the answer, and the MLP must supply it.


**Exercise.** Why can the "Paris" unit not be a linear function of the two features, without the ReLU?

<details><summary>Answer</summary>

Without the ReLU the unit would output $2\,\text{country} + 2\,\text{capital} - 1.5$, which is never switched off: with "a capital" but no France, $(0.13, 0.55)$, it gives $-0.14$, and more "capital" alone keeps raising it. The ReLU turns the unit into a switch: it is 0 below the threshold and positive above it. Since each feature is at most about 0.6 here, the sum passes 1.5 only when both are present, so the unit fires for the combination and stays silent otherwise.

</details>


## Part 7 — All tokens at once

Every token has a query, not only *capital*. We collect the queries in a
matrix $Q$ (one row per token), the keys in $K$ and the values in $V$. Then
all the scores are one matrix product, and the whole computation is

$$\mathrm{Attn}(Q, K, V) = \mathrm{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V ,$$

where the softmax is applied to each row. Row $i$ of the weight matrix says
how much token $i$ takes from each token. The queries of *France* and *its*
below are chosen arbitrarily; the row of *capital* is the computation of
Parts 2 to 4.

Here every token can take information from every token, including tokens
that come later in the text. We restrict this later, when the model is
trained to predict the next token.


In [ ]:
Q = np.array([[0., 1.],    # France
              [1., 1.],    # its
              [1., 0.]])   # capital, as above
P = softmax(Q @ K.T / np.sqrt(d_k))
OUT = P @ V

fig, ax = plt.subplots(figsize=(4.2, 3.4))
ax.imshow(P, cmap="Blues", vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{P[i, j]:.2f}", ha="center", va="center",
                color="white" if P[i, j] > 0.5 else "black")
ax.set_xticks(range(3), TOKENS); ax.set_yticks(range(3), TOKENS)
ax.set_xlabel("key (the token read)"); ax.set_ylabel("query (the token reading)")
ax.set_title("the attention weights, one row per token")
plt.tight_layout(); save(fig, "fig-t-w5-h1l1-p7"); plt.show()

print("row sums:", P.sum(1))
print("capital row = part 4:", np.allclose(OUT[2], out))


## Try it yourself

1. Change the query of *capital* to $[0, 1]$, so that it looks for a
   function word. Which token now receives the largest weight?
2. Multiply the query by 5, to $[5, 0]$. What happens to the weights? (This
   is the effect that the division by $\sqrt{d_k}$ controls.)
3. Replace the value of *France* by $[0, 0]$. Do the weights change? Does the
   output?

<details><summary>Answers</summary>

1. The weights become 0.14 for *France*, 0.58 for *its* and 0.28 for
   *capital*, and the output is $[0.42,\ 0.86]$: mostly function-word
   information.
2. The weights become 0.97, 0.001 and 0.03, and the output is about
   $[1.00,\ 0.03]$. A larger query spreads the scores further apart, and the
   softmax puts almost all the weight on one key. The division by
   $\sqrt{d_k}$ keeps this in check.
3. The weights do not change (0.58, 0.14, 0.28), because the values do not
   enter the scores. The output becomes $[0.28,\ 0.42]$.

</details>
